# Effect Sizes, Multiple Testing and How Data Fools You

Notebook for **Chapter 5** of the *stats* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/stats/ch05.html

Tip: change a number, predict the output, then run the cell again.

## 5.1 Effect Sizes: How Big Is the Difference?

In [ ]:
import numpy as np
from scipy import stats
n1, m1, s1, n2, m2, s2 = 40, 250, 60, 45, 225, 50
sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
d = (m1 - m2) / sp
g = d * (1 - 3 / (4 * (n1 + n2) - 9))
se = np.sqrt((n1 + n2) / (n1 * n2) + d**2 / (2 * (n1 + n2)))
print("pooled sd %.2f  d %.3f  hedges g %.3f  SE(d) %.3f" % (sp, d, g, se))
print("95%% CI for d: (%.3f, %.3f)" % (d - 1.96 * se, d + 1.96 * se))
print("probability of superiority: %.3f" % stats.norm.cdf(d / np.sqrt(2)))

pooled sd 54.93  d 0.455  hedges g 0.451  SE(d) 0.220
95% CI for d: (0.024, 0.887)
probability of superiority: 0.626


In [ ]:
n, diff, sd = 2_000_000, 0.5, 100
se = sd * np.sqrt(2 / n)
print("SE %.3f  z %.2f  p %.2e  d %.4f" % (se, diff / se, 2 * stats.norm.sf(diff / se), diff / sd))

SE 0.100  z 5.00  p 5.73e-07  d 0.0050


In [ ]:
a, b, c, dd = 600, 9400, 500, 9500     # B convert, B not, A convert, A not
pA, pB = c / (c + dd), a / (a + b)
OR = (a / b) / (c / dd); se = np.sqrt(1/a + 1/b + 1/c + 1/dd)
print("RD %.3f  RR %.3f  OR %.4f" % (pB - pA, pB / pA, OR))
print("95%% CI OR: (%.3f, %.3f)" % (np.exp(np.log(OR) - 1.96 * se), np.exp(np.log(OR) + 1.96 * se)))
# rare vs common event: OR vs RR
for pa, pb in [(0.05, 0.06), (0.40, 0.48)]:
    print("pA=%.2f pB=%.2f RR=%.3f OR=%.3f" % (pa, pb, pb / pa, (pb / (1 - pb)) / (pa / (1 - pa))))

RD 0.010  RR 1.200  OR 1.2128
95% CI OR: (1.073, 1.370)
pA=0.05 pB=0.06 RR=1.200 OR=1.213
pA=0.40 pB=0.48 RR=1.200 OR=1.385


## 5.2 Multiple Testing: Bonferroni, Holm and Benjamini-Hochberg

In [ ]:
def bonferroni(p):
    p = np.asarray(p); return np.minimum(p * len(p), 1)
def holm(p):
    p = np.asarray(p); m = len(p); o = np.argsort(p)
    run = 0; out = np.empty(m)
    for rank, i in enumerate(o):
        run = max(run, (m - rank) * p[i]); out[i] = min(run, 1)
    return out
def bh(p):
    p = np.asarray(p); m = len(p); o = np.argsort(p)
    adj = p[o] * m / (np.arange(m) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(m); out[o] = np.minimum(adj, 1); return out
p = [0.001, 0.008, 0.020, 0.041, 0.300]
print("Bonferroni:", np.round(bonferroni(p), 3))
print("Holm      :", np.round(holm(p), 3))
print("BH        :", np.round(bh(p), 4))
print("scipy BH  :", np.round(stats.false_discovery_control(p), 4))
print("rejected at 0.05 -> Bonf", (bonferroni(p) <= .05).sum(), " Holm", (holm(p) <= .05).sum(), " BH", (bh(p) <= .05).sum())

Bonferroni: [0.005 0.04  0.1   0.205 1.   ]
Holm      : [0.005 0.032 0.06  0.082 0.3  ]
BH        : [0.005  0.02   0.0333 0.0512 0.3   ]
scipy BH  : [0.005  0.02   0.0333 0.0512 0.3   ]
rejected at 0.05 -> Bonf 2  Holm 2  BH 3


In [ ]:
rng = np.random.default_rng(40)
m, k, n = 1000, 100, 40
true = np.zeros(m, bool); true[:k] = True
a = rng.normal(0, 1, (m, n)); b = rng.normal(0, 1, (m, n)) + np.where(true, 0.8, 0.0)[:, None]
pv = stats.ttest_ind(a, b, axis=1).pvalue
rules = {"no correction": pv <= 0.05, "Bonferroni": bonferroni(pv) <= 0.05, "Holm": holm(pv) <= 0.05, "BH (q=0.05)": bh(pv) <= 0.05}
print(f"{'rule':16}{'discoveries':>12}{'true':>6}{'false':>7}{'FDP':>8}{'power':>8}")
for name, rej in rules.items():
    tp = (rej & true).sum(); fp = (rej & ~true).sum()
    print(f"{name:16}{rej.sum():12d}{tp:6d}{fp:7d}{fp / max(rej.sum(), 1):8.3f}{tp / k:8.2f}")

rule             discoveries  true  false     FDP   power
no correction            150    94     56   0.373    0.94
Bonferroni                29    29      0   0.000    0.29
Holm                      29    29      0   0.000    0.29
BH (q=0.05)               79    76      3   0.038    0.76


In [ ]:
# plot
import matplotlib.pyplot as plt
o = np.sort(pv); ranks = np.arange(1, m + 1)
plt.figure(figsize=(6.2, 3.8))
plt.plot(ranks[:250], o[:250], ".", color="#1f77b4", label="sorted p-values")
plt.plot(ranks[:250], 0.05 * ranks[:250] / m, color="#d62728", label="BH line: 0.05 k/m")
plt.axhline(0.05 / m, color="gray", ls="--", label="Bonferroni: 0.05/m")
plt.axhline(0.05, color="black", ls=":", label="no correction")
plt.xlabel("rank k of the p-value"); plt.ylabel("p-value"); plt.ylim(0, 0.12); plt.legend(fontsize=7)
plt.title("1,000 tests: three decision lines")
plt.tight_layout(); plt.savefig(f"{IMG}/ch05_multiple_testing.png", dpi=120); plt.close()

## 5.3 p-Hacking: Peeking, Slicing and Forking Paths

In [ ]:
rng = np.random.default_rng(41)
S, looks, step, p0 = 5000, 10, 1000, 0.05
xa = np.cumsum(rng.binomial(step, p0, (S, looks)), axis=1)
xb = np.cumsum(rng.binomial(step, p0, (S, looks)), axis=1)
nn = step * np.arange(1, looks + 1)
pp = (xa + xb) / (2 * nn)
z = (xb - xa) / nn / np.sqrt(np.maximum(pp * (1 - pp) * 2 / nn, 1e-12))
pvals = 2 * stats.norm.sf(np.abs(z))
print("single look at the end        : false positive rate = %.3f" % (pvals[:, -1] < .05).mean())
print("peek at all 10 looks, stop on p<.05: false positive rate = %.3f" % (pvals < .05).any(axis=1).mean())
print("Bonferroni over 10 looks (0.005 each): %.3f" % (pvals < .005).any(axis=1).mean())

single look at the end        : false positive rate = 0.052
peek at all 10 looks, stop on p<.05: false positive rate = 0.196
Bonferroni over 10 looks (0.005 each): 0.024


In [ ]:
S, G = 5000, 20
pv_sub = rng.uniform(0, 1, (S, G))             # null p-values are uniform
print("theory P(at least one): %.4f" % (1 - 0.95**G))
print("simulated, uncorrected : %.4f   mean number significant: %.2f" % ((pv_sub < .05).any(1).mean(), (pv_sub < .05).sum(1).mean()))
holm_any = np.array([(holm(row) <= .05).any() for row in pv_sub[:2000]])
print("simulated, Holm        : %.4f" % holm_any.mean())

theory P(at least one): 0.6415
simulated, uncorrected : 0.6424   mean number significant: 1.01
simulated, Holm        : 0.0460


## 5.4 More Ways Data Fools You: Winner's Curse, Regression to the Mean, and Low Prior Odds

In [ ]:
rng = np.random.default_rng(42)
obs_top, true_top = [], []
for _ in range(2000):
    true = rng.normal(0, 0.3, 200); obs = true + rng.normal(0, 0.5, 200)
    top = np.argsort(obs)[-5:]
    obs_top.append(obs[top].mean()); true_top.append(true[top].mean())
print("top-5 measured lift: %.2f   top-5 true lift: %.2f   shrink factor %.2f" % (np.mean(obs_top), np.mean(true_top), 0.09 / 0.34))

top-5 measured lift: 1.34   top-5 true lift: 0.35   shrink factor 0.26


In [ ]:
rng = np.random.default_rng(43)
true = rng.normal(200, 20, 500)
d1 = true + rng.normal(0, 30, 500); d2 = true + rng.normal(0, 30, 500)
worst = np.argsort(d1)[-25:]
print("flagged branches: day 1 mean %.1f -> day 2 mean %.1f (nothing was changed)" % (d1[worst].mean(), d2[worst].mean()))
print("theory for day 2: %.1f" % (200 + 20**2 / (20**2 + 30**2) * (d1[worst].mean() - 200)))

flagged branches: day 1 mean 272.3 -> day 2 mean 224.9 (nothing was changed)
theory for day 2: 222.2


In [ ]:
def ppv(power, prior, alpha=0.05):
    return power * prior / (power * prior + alpha * (1 - prior))
for pw, pr in [(0.8, 0.1), (0.3, 0.1), (0.8, 0.02), (0.8, 0.5)]:
    print("power=%.1f prior=%.2f -> PPV=%.3f" % (pw, pr, ppv(pw, pr)))

power=0.8 prior=0.10 -> PPV=0.640
power=0.3 prior=0.10 -> PPV=0.400
power=0.8 prior=0.02 -> PPV=0.246
power=0.8 prior=0.50 -> PPV=0.941


## 5.5 An Honest Reporting Checklist

## Chapter recap

## Practice set

Problem 5.6.1 Means 80 and 70, pooled SD 20. What is d, and is it small, medium or large?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/stats/ch05.html